## 9 - Merging & Joining

### concat() - stack DataFrames
```python
pd.concat([df1, df2])            # stack vertically (more rows)
pd.concat([df1, df2], axis=1)    # stack horizontally (more columns)
pd.concat([df1, df2], ignore_index=True)   # reset index
```

### merge() - SQL-style joins
```python
pd.merge(left, right, on='key')            # inner join on key column
pd.merge(left, right, on='key', how='left')   # left join
pd.merge(left, right, on='key', how='right')  # right join
pd.merge(left, right, on='key', how='outer')  # full outer join
```

### Join types visualised
```
Inner  → only rows where key exists in BOTH  (intersection)
Left   → all rows from LEFT, NaN where right has no match
Right  → all rows from RIGHT, NaN where left has no match
Outer  → ALL rows from BOTH, NaN where match missing  (union)
```

### When to use which
| Task | Use |
|------|-----|
| Same columns, more rows | `concat(axis=0)` |
| Same rows, more columns | `concat(axis=1)` or `join()` |
| Combine on a key column | `merge()` |
| Combine on index | `join()` |


In [6]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv('students_enriched.csv')

In [8]:
# concat() - stack more rows
batch1 = df.iloc[:25].copy()
batch2 = df.iloc[25:].copy()

combined = pd.concat([batch1, batch2], ignore_index=True)
print(f'batch1: {len(batch1)}  batch2: {len(batch2)}  combined: {len(combined)}')

batch1: 25  batch2: 25  combined: 50


In [9]:
# Create two tables to merge 

# Table 1: Student academic info
students = df[['student_id','name','gender','study_category','maths','average','grade']].copy()

# Table 2: Extracurricular activities
np.random.seed(1)
extra = pd.DataFrame({
    'student_id': df['student_id'].sample(40, random_state=1).values,
    'activity': np.random.choice(['Sports','Music','Debate','Art','Coding'], 40),
    'hours_week': np.round(np.random.uniform(1,8,40),1)
})

print('students shape:', students.shape)
print('extra shape:', extra.shape)

students shape: (50, 7)
extra shape: (40, 3)


In [12]:
# Inner join
inner = pd.merge(students, extra, on='student_id', how='inner')

print(f'Inner join: {len(inner)} rows (only students WITH activities)')
print(inner.head())

Inner join: 40 rows (only students WITH activities)
  student_id    name  gender study_category  maths  average grade activity  \
0       S002   Aanya    Male         Medium   57.5    76.28     B    Music   
1       S003   Aditi  Female         Medium   78.7    63.95     C      Art   
2       S004   Arjun  Female         Medium   69.9    69.55     C   Sports   
3       S005  Bhavna  Female           High   57.1    64.55     C    Music   
4       S007   Deepa  Female         Medium   66.5    65.75     C   Sports   

   hours_week  
0         7.2  
1         3.9  
2         1.2  
3         2.8  
4         7.2  


In [14]:
# Left join
left  = pd.merge(students, extra, on='student_id', how='left')

print(f'Left join: {len(left)} rows (all 50 students, NaN if no activity)')
print('Students without activity:', left['activity'].isna().sum())

print(left[left['activity'].isna()][['name','activity']].head())

Left join: 50 rows (all 50 students, NaN if no activity)
Students without activity: 10
      name activity
0    Aarav      NaN
5   Chirag      NaN
8    Divya      NaN
9     Esha      NaN
11   Gauri      NaN


In [16]:
# outer join 
outer = pd.merge(students, extra, on='student_id', how='outer')

print(f'Outer join: {len(outer)} rows')
print(outer.head())

Outer join: 50 rows
  student_id    name  gender study_category  maths  average grade activity  \
0       S001   Aarav  Female            Low   52.9    69.55     C      NaN   
1       S002   Aanya    Male         Medium   57.5    76.28     B    Music   
2       S003   Aditi  Female         Medium   78.7    63.95     C      Art   
3       S004   Arjun  Female         Medium   69.9    69.55     C   Sports   
4       S005  Bhavna  Female           High   57.1    64.55     C    Music   

   hours_week  
0         NaN  
1         7.2  
2         3.9  
3         1.2  
4         2.8  


In [18]:
# concat horizontally 
part_a = df[['student_id','name','maths']].head(5)
part_b = df[['science','english']].head(5)

h_concat = pd.concat([part_a, part_b], axis=1)
print('Horizontal concat:')
print(h_concat)

Horizontal concat:
  student_id    name  maths  science    english
0       S001   Aarav   52.9     56.0  69.300000
1       S002   Aanya   57.5     88.1  80.600000
2       S003   Aditi   78.7     57.8  77.700000
3       S004   Arjun   69.9     63.8  71.895652
4       S005  Bhavna   57.1     78.6  61.800000
